In [ ]:
# UNIFIED EVALUATION Prompts for both Question & Answer


## Question Evaluation Prompt

def u_Build_Question_Evaluator(
    *,
    criterion,
    context,
    image_list,
    task_metadata,
    question,
    page_reference="",
    section_hint=""
):
    c = U_CRITERIA[criterion]
    cot_steps = "\n".join([f"- {s}" for s in c["cot"]])

    return f"""You are an expert evaluator specializing in Software Requirements Specification (SRS) documents and question quality assessment.

Evaluate ONLY the quality of the question. Be strict, consistent, and evidence-based. Use the criterion below.

CRITERION NAME: {c["name"]}
CRITERION DESCRIPTION: {c["definition"]}

RUBRIC (score must be 0, 1, or 2):
- 0 (Poor): {c["rubric"][0]}
- 1 (Fair): {c["rubric"][1]}
- 2 (Excellent): {c["rubric"][2]}

EVALUATION STEPS:
{cot_steps}

INPUTS:
[Question] {question}

[Supporting Context / Evidence]
Context: {context}
Extracted Images: {image_list}
Page Reference: {page_reference}
Section Hint: {section_hint}
Task Metadata: {task_metadata}

OUTPUT:
Return valid JSON only, with exactly these keys:
{{
  "score": <0, 1, or 2>,
  "reason": "<one evidence-based sentence>"
}}

INSTRUCTIONS:
- Judge ONLY the quality of the question for the specified criterion.
- Use the evaluation steps internally, but do not output them.
- Use the supporting context, page reference, section hint, images, and task metadata.
- Do not invent requirement IDs, page numbers, figures, tables, section names, or named components that are not provided in the input.
- If images are listed but no image content or description is provided, do not infer visual evidence from filenames alone.
- The reason must be exactly one sentence, about 25 to 100 words.
- The reason must justify why the selected score is appropriate.
- When concrete evidence exists, mention at least one specific anchor such as a requirement ID, appendix, section, page, figure, table, or named component.
- Avoid vague or generic wording like "supported by the text" when a more specific explanation is possible, unless it is connected to a specific evidence anchor.
- Do not include markdown, code fences, explanations, or extra JSON keys.
"""


def u_extract_subtask_definition(full_definition, subtask_name):
    if subtask_name in full_definition:
        start_idx = full_definition.find(subtask_name)
        match = re.compile(r'\n\d+\.\s').search(full_definition, start_idx + len(subtask_name))
        return full_definition[start_idx:match.start()].strip() if match else full_definition[start_idx:].strip()
    return full_definition


def u_resolve_image_paths(raw_images_str):
    if not isinstance(raw_images_str, str) or raw_images_str.strip() in ("", "None", "[]"):
        return []
    try:
        img_list = ast.literal_eval(raw_images_str)
        if not isinstance(img_list, list):
            return []
    except:
        return []
    paths = []
    for img_path in img_list:
        p = Path(str(img_path))
        candidates = [p, BASE_PATH / p, U_IMAGE_DIR / p.name, BASE_PATH / "extracted_images" / p.name]
        for candidate in candidates:
            if candidate.exists():
                paths.append(candidate)
                break
    return paths



## Answer Evaluation Prompt

In [ ]:
def u_Build_Answer_Evaluator(
    *,
    criterion,
    question,
    answer,
    context,
    image_list="None",
    citations="",
    difficulty_label="",
    task_metadata="",
    page_reference=None,
    section_hint=""
):
    c = U_CRITERIA[criterion]
    cot_steps = "\n".join([f"- {s}" for s in c["cot"]])

    # Keep backward compatibility with existing U5/U10 calls:
    # current callers pass `citations=page_ref`, where page_ref already contains section hint + page reference.
    if page_reference is None:
        page_reference = citations

    return f"""You are an expert evaluator specializing in Software Requirements Specification (SRS) documents and answer quality assessment.

Evaluate ONLY the quality of the answer. Be strict, consistent, and evidence-based. Use the criterion below.

CRITERION NAME: {c["name"]}
CRITERION DESCRIPTION: {c["definition"]}

RUBRIC (score must be 0, 1, or 2):
- 0 (Poor): {c["rubric"][0]}
- 1 (Fair): {c["rubric"][1]}
- 2 (Excellent): {c["rubric"][2]}

EVALUATION STEPS:
{cot_steps}

INPUTS:
[Question] {question}

[Answer] {answer}

[Supporting Context / Evidence]
Context: {context}
Extracted Images: {image_list}
Page Reference: {page_reference}
Section Hint: {section_hint}
Difficulty Label: {difficulty_label}
Task Metadata: {task_metadata}

OUTPUT:
Return valid JSON only, with exactly these keys:
{{
  "score": <0, 1, or 2>,
  "reason": "<one evidence-based sentence>"
}}

INSTRUCTIONS:
- Judge ONLY the quality of the answer for the specified criterion.
- Use the evaluation steps internally, but do not output them.
- Use the supporting context, page reference, section hint, images, and task metadata only as evidence.
- Do not invent requirement IDs, page numbers, figures, tables, section names, or named components that are not provided in the input.
- If images are listed but no image content or description is provided, do not infer visual evidence from filenames alone.
- The reason must be exactly one sentence, about 25 to 100 words.
- The reason must justify why the selected score is appropriate.
- When concrete evidence exists, mention at least one specific anchor such as a requirement ID, appendix, section, page, figure, table, or named component.
- Avoid vague or generic wording like "supported by the text" when a more specific explanation is possible, unless it is connected to a specific evidence anchor.
- Do not include markdown, code fences, explanations, or extra JSON keys.
"""
